In [58]:
import pandas as pd
import numpy as np
import joblib
from tqdm import tqdm
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

ind_hosp = pd.read_parquet('ind_hosp.parquet')
calibrated = joblib.load('lightgbm.pkl')

cols_to_drop = [ 
    'dischtime', 
    'current_date',
    'target_readmission_30d',
    'los'
]
cols_to_drop = [c for c in cols_to_drop if c in ind_hosp.columns]

X = ind_hosp.drop(columns=cols_to_drop)
bool_cols = X.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)

patient_target = ind_hosp.groupby('subject_id')['target_readmission_30d'].max().reset_index()
patient_target.columns = ['subject_id', 'has_readmission']
train_val_ids, test_ids = train_test_split(
    patient_target['subject_id'],
    test_size=0.2,
    random_state=42,
    stratify=patient_target['has_readmission']
)

test_patients = test_ids.tolist()
#test_patients =test_patients[:1000]
print(f"Test patients: {len(test_patients)}")

Test patients: 35915


In [59]:
icd_cols = [col for col in X.columns if col.startswith('icd_')]
ccsr_cols = [col for col in X.columns if col.startswith('ccsr_')]
lab_cols = [col for col in X.columns if col.startswith('lab_') and col.endswith('_daily')]

features_to_analyze = (
    icd_cols +
    ccsr_cols +
    lab_cols +
    [
        'num_diagnoses',
        'num_chronic',
        'comorbidity_score',
        'num_medications_daily',
        'prior_admissions_12mo',
        'cumulative_procedures',
        'cumulative_medications',
        'num_procedures_daily',
        'gender_male',
        'age'
    ]
)

features_to_analyze = [f for f in features_to_analyze if f in X.columns]
print(f"Analyzing {len(features_to_analyze)} features")

Analyzing 60 features


In [60]:
clinical_norms = {
    'lab_50983_daily': 141,
    'lab_50971_daily': 4.2,
    'lab_50902_daily': 102,
    'lab_50882_daily': 27,
    
    'lab_50912_daily': 0.85,
    'lab_51006_daily': 13,
    
    'lab_50931_daily': 85,
    'lab_50893_daily': 9.35,
    
    'lab_50868_daily': 14,
    'lab_51222_daily': 15.6,
    'lab_51301_daily': 7,
    'lab_51265_daily': 275,
    'lab_51221_daily': 45.5,
    'lab_51250_daily': 90,
    'lab_51277_daily': 13,

    'lab_50960_daily': 2.1,
    'lab_50970_daily': 3.6,
    'lab_51248_daily': 39,
    'lab_51249_daily': 34.5,
    'lab_51279_daily': 5.35
}

In [ ]:
def get_counterfactual_value(feature, patient_value, X):
    if feature.startswith('lab_'):
        return clinical_norms.get(feature, X[feature].mean())
    
    if feature == 'age':
        return X['age'].mean() 
    
    if feature == 'gender_male':
        return 1 if patient_value == 0 else 0
    
    if feature in ['num_medications_daily', 'num_procedures_daily']:
        return 0
    
    if feature in ['cumulative_medications', 'cumulative_procedures']:
        return 0
    
    if feature in ['prior_admissions_12mo', 'comorbidity_score']:
        return 0
    
    if feature.startswith('icd_') or feature.startswith('ccsr_'):
        return 0
    
    if feature in ['num_diagnoses', 'num_chronic']:
        return X[feature].mean()
    
    return X[feature].mean()

def predict_risk(model, data):
    p_t = model.predict_proba(data)[:, 1]
    risk = 1 - np.prod(1 - p_t)
    return risk

In [62]:
all_results = []

for patient_id in tqdm(test_patients, desc="Patients"):
    patient_data = X[X['subject_id'] == patient_id].copy()
    for hadm_id in patient_data['hadm_id'].unique():
        hadm_data = patient_data[patient_data['hadm_id'] == hadm_id].copy()
        hadm_data = hadm_data.drop(['subject_id', 'hadm_id'], axis=1)
        risk_actual = predict_risk(calibrated, hadm_data)
        
        for feature in features_to_analyze:
            if feature not in hadm_data.columns:
                continue
            
            hadm_counter = hadm_data.copy()
            hadm_counter[feature] = get_counterfactual_value(feature, hadm_data[feature].iloc[0], X)
            
            risk_counter = predict_risk(calibrated, hadm_counter)
            delta = risk_actual - risk_counter
            
            all_results.append({
                'subject_id': patient_id,
                'hadm_id': hadm_id,
                'feature': feature,
                'delta': delta,
                'risk_actual': risk_actual,
                'risk_counterfactual': risk_counter,
                'n_days': len(hadm_data)
            })

results_df = pd.DataFrame(all_results)

Patients: 100%|██████████| 35915/35915 [1:59:14<00:00,  5.02it/s]  


In [ ]:
patient_avg = results_df.groupby(['subject_id', 'feature'], as_index=False).agg({
    'delta': 'mean',
    'risk_actual': 'mean',
    'n_days': 'mean'
})

feature_stats = patient_avg.groupby('feature').agg({
    'delta': ['mean', 'std', 'min', 'max', 'count'],
    'risk_actual': 'mean',
    'n_days': 'mean'
}).round(4)

print(f"\nReadmissions analyzed {len(results_df)}")
print(f"feature_stats: {len(feature_stats)}")

feature_stats.columns = [
    'delta_mean', 
    'delta_std', 
    'delta_min', 
    'delta_max', 
    'n_patients', 
    'risk_mean',
    'n_days_mean'
]
feature_stats = feature_stats.sort_values('delta_mean', ascending=False)

results_df.to_csv('total_deltas.csv', index=False)
feature_stats.to_csv('delta_summary.csv')

print("\nFactors that increase risk (Δ > 0):")
positive = feature_stats[feature_stats['delta_mean'] > 0].sort_values('delta_mean', ascending=False).head(10)
for feature, row in positive.iterrows():
    print(f"  {feature}: Δ = {row['delta_mean']:+.4f} ± {row['delta_std']:.4f}")

print("\nFactors that decrease risk (Δ < 0):")
negative = feature_stats[feature_stats['delta_mean'] < 0].sort_values('delta_mean', ascending=True).head(10)
for feature, row in negative.iterrows():
    print(f"  {feature}: Δ = {row['delta_mean']:+.4f} ± {row['delta_std']:.4f}")


Readmissions analyzed 4896960
feature_stats: 60

Factors that increase risk (Δ > 0):
  lab_50983_daily: Δ = +0.0256 ± 0.0140
  comorbidity_score: Δ = +0.0203 ± 0.0396
  prior_admissions_12mo: Δ = +0.0193 ± 0.0432
  lab_51248_daily: Δ = +0.0155 ± 0.0211
  lab_51221_daily: Δ = +0.0135 ± 0.0135
  lab_50882_daily: Δ = +0.0109 ± 0.0077
  lab_51265_daily: Δ = +0.0084 ± 0.0097
  age: Δ = +0.0043 ± 0.0292
  lab_51301_daily: Δ = +0.0026 ± 0.0064
  ccsr_BLD003: Δ = +0.0017 ± 0.0050

Factors that decrease risk (Δ < 0):
  lab_50931_daily: Δ = -0.0445 ± 0.0296
  lab_51250_daily: Δ = -0.0417 ± 0.0382
  num_diagnoses: Δ = -0.0174 ± 0.0186
  lab_50971_daily: Δ = -0.0156 ± 0.0137
  num_medications_daily: Δ = -0.0148 ± 0.0262
  cumulative_medications: Δ = -0.0129 ± 0.0209
  num_chronic: Δ = -0.0122 ± 0.0200
  lab_50893_daily: Δ = -0.0100 ± 0.0073
  lab_50960_daily: Δ = -0.0090 ± 0.0116
  lab_51279_daily: Δ = -0.0082 ± 0.0119
